# Create a Langchain Agent with MCP tools

In [16]:
# Creare environment
import os 
from dotenv import load_dotenv 
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


# Get LLM model 
from langchain.chat_models import init_chat_model 

gemma = init_chat_model(model="gemma4:latest", model_provider="ollama")
primary_llm = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")

fallback_llm_1 = init_chat_model(model="gpt-5.4-nano", model_provider="openai",
                 model_kwargs={"temperature": 0.5, "max_tokens": 1000})
fallback_llm_2 = init_chat_model(model="gpt-5.4-mini", model_provider="openai",
                 model_kwargs={"temperature": 0.5, "max_tokens": 1000})

/Users/nali/Documents/YTLLMs/.venv/lib/python3.13/site-packages/langchain/chat_models/base.py:496: UserWarning: Parameters {'temperature', 'max_tokens'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


## Create the MCP Client 

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [66]:
client = MultiServerMCPClient({
    "open_meteo":{
        "transport":"stdio",
        "command":"npx",
        "args":[
            "-y",
            "open-meteo-mcp-server"
        ]
    },
    
    "tavily_search":{
        "transport":"stdio",
        "command":"npx",
        "args":[
            "-y","tavily-mcp@latest"
        ]
    },

    "file_system":{
        "transport":"stdio",
        "command":"npx",
        "args":[
            "-y",
            "@modelcontextprotocol/server-filesystem", 
            "/Users/nali/Documents/Devs"
        ]
    }
    
})

In [67]:
tools = await client.get_tools()

In [68]:
for tool in tools: 
    print(tool.name)
    #

weather_forecast
weather_archive
air_quality
marine_weather
elevation
flood_forecast
geocoding
dwd_icon_forecast
gfs_forecast
meteofrance_forecast
ecmwf_forecast
jma_forecast
metno_forecast
gem_forecast
seasonal_forecast
climate_projection
ensemble_forecast
tavily_search
tavily_extract
tavily_crawl
tavily_map
tavily_research
read_file
read_text_file
read_media_file
read_multiple_files
write_file
edit_file
create_directory
list_directory
list_directory_with_sizes
directory_tree
move_file
search_files
get_file_info
list_allowed_directories


## Create a langchain agent with mcp tool

In [69]:
from langchain.agents import create_agent 

prompt= """You are a helpful asssistant.
use open_meteo tool for providing weather related query.
Use tavily_search tool for current news and affairs.
""" 


agent = create_agent(
    model=fallback_llm_1, 
    tools=tools,
    system_prompt=prompt
)

## Invoke the agent 

In [79]:
user_query = "Delete Devs/README.md file. Confirm delete"

In [80]:
from langchain.messages import HumanMessage, SystemMessage 

try: 
    response = await agent.ainvoke({
        "messages":[
            HumanMessage(content=user_query)
        ]
    })
except Exception as e: 
    print(f"Invoke error. {e}")

In [81]:
print(response["messages"][-1].content)

Confirmed: **Dev/README.md** has been deleted (moved to **Dev/README.md.deleted**).
